In [0]:
%pip install mlflow torch transformers pandas==2.2.2 numpy==1.26.4

In [0]:
import mlflow
import pandas as pd
import transformers
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, pipeline
from mlflow.models.signature import infer_signature

# --- Configuración (sin cambios) ---
mlflow.set_registry_uri("databricks-uc")
model_name = "google/flan-t5-base"
registered_model_name = "bluetab.rag.flan_t5_base_model"

# --- Carga del modelo (sin cambios) ---
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
pipe = pipeline("text2text-generation", model=model, tokenizer=tokenizer, device=-1)

# --- Ejemplo de entrada (sin cambios) ---
contexto = "La paella valenciana es un plato de arroz originario de la Comunidad Valenciana."
pregunta = "¿De dónde es la paella?"
prompt_completo = f"Contexto: {contexto}\n\nPregunta: {pregunta}\n\nRespuesta:"
input_data = [[{'role': 'user', 'content': prompt_completo}]]
input_example = pd.DataFrame({"messages": input_data})

# --- CAMBIO #1: Formato del ejemplo de salida para coincidir con la API de Chat ---
prompts_para_ejemplo = [chat_history[-1]['content'] for chat_history in input_example["messages"].tolist()]
texto_prediccion = pipe(prompts_para_ejemplo[0], max_length=50)[0]["generated_text"]
# Se construye la estructura anidada exacta que LangChain espera
output_example = {
    "choices": [
        {
            "message": {
                "role": "assistant",
                "content": texto_prediccion
            }
        }
    ]
}
signature = infer_signature(input_example, pd.DataFrame(output_example))


# --- Wrapper con la estructura de salida final ---
class FlanT5Wrapper(mlflow.pyfunc.PythonModel):
    def load_context(self, context):
        from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, pipeline
        model_to_load = "google/flan-t5-base"
        self.tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-base")
        self.model = AutoModelForSeq2SeqLM.from_pretrained(model_to_load)
        self.pipe = pipeline("text2text-generation", model=self.model, tokenizer=self.tokenizer, device=-1)

    def predict(self, context, model_input):
        prompts = [chat_history[-1]['content'] for chat_history in model_input["messages"].tolist()]
        
        # Generamos la lista de textos como antes
        predictions_text = [self.pipe(p, max_length=150, temperature=0.1)[0]["generated_text"] for p in prompts]
        
        # --- CAMBIO #2: Construimos la estructura de salida final ---
        choices = []
        for text in predictions_text:
            choices.append(
                {
                    "message": {
                        "role": "assistant",
                        "content": text
                    }
                }
            )
        
        # Devolvemos el diccionario con la clave "choices"
        return {"choices": choices}



In [0]:
print(output_example)

In [0]:
# --- Registro de la nueva versión del modelo ---
with mlflow.start_run(run_name="Register Flan-T5 Model (Final LangChain Compatible Output)") as run:
    mlflow.pyfunc.log_model(
        artifact_path="flan_t5_base_model_final",
        python_model=FlanT5Wrapper(),
        registered_model_name=registered_model_name,
        input_example=input_example,
        signature=signature,
        pip_requirements=[
            f"mlflow=={mlflow.__version__}", "torch", "transformers",
            "pandas", "numpy"
        ]
    )

print(f"Versión final del modelo '{registered_model_name}' ha sido registrada.")

In [0]:
import mlflow.pyfunc

# Cargar el modelo registrado
model_version = 2  # Especificar la versión del modelo si es necesario
model = mlflow.pyfunc.load_model(f"models:/{registered_model_name}/{model_version}")

In [0]:
# 4. Se crea un ejemplo de entrada y salida con el formato RAG
contexto1 = "La paella valenciana es un plato de arroz originario de la Comunidad Valenciana. Sus ingredientes tradicionales incluyen arroz, pollo, conejo y verduras locales."
pregunta1 = "¿Qué ingredientes lleva la paella?"
prompt_rag1 = f"Contexto: {contexto1}\n\nPregunta: {pregunta1}\n\nResponde basándote únicamente en el contexto.\nRespuesta:"

contexto2 = "El gazpacho es una sopa fría originaria de Andalucía. Sus ingredientes principales son tomate, pepino, pimiento, cebolla, ajo, aceite de oliva, vinagre y pan."
pregunta2 = "¿Qué ingredientes lleva el gazpacho?"
prompt_rag2 = f"Contexto: {contexto2}\n\nPregunta: {pregunta2}\n\nResponde basándote únicamente en el contexto.\nRespuesta:"

input_example = pd.DataFrame({"prompt": [prompt_rag1, prompt_rag2]})

# Realizar la predicción
prediction = model.predict(input_example)

# Mostrar el resultado de la predicción
display(prediction)